In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

In [3]:
import pandas as pd

df = pd.read_csv("notebooks/outputs/baskets_dogs_transcripts.csv")
df

,object_category,round_ix,pair_id,transcript,llm_extracted_object_descriptions,llm_extracted_object_descriptions_updated,llm_extracted_object_descriptions_GPT_5
0,baskets,1,Pair_20,"D: Alright, the first one is a, uh--is a very ...","{\n ""object_#1"": ""a very long basket"",\n ...","{\n ""object_#1"": ""very long basket, rectang...","{\n ""object_#1"": ""a very long basket, recta..."
1,baskets,1,Pair_21,"D: Ok, uh…\n You ready?\n \n M: Yup\n \n D: (l...","{\n ""object_#1"": ""a long basket"",\n ""obj...","{\n ""object_#1"": ""a long basket, It looks l...","{\n ""object_#1"": ""long basket, looks like a..."
2,baskets,1,Pair_22,"D: Alright, hello?\n Um, the first one is--is....","{\n ""object_#1"": ""It's rectangular, and it'...","{\n ""object_#1"": ""rectangular, it's a baske...","{\n ""object_#1"": ""rectangular, light brown,..."
3,baskets,1,Pair_23,"D: Ok, Parastoo you see the...sort of...oblong...","{\n ""object_#1"": ""oblong, rectangular baske...","{\n ""object_#1"": ""oblong, rectangular baske...","{\n ""object_#1"": ""oblong, rectangular baske..."
4,baskets,1,Pair_24,"D: K, uh, the first basket is longer than the ...","{\n ""object_#1"": ""One handle, long basket"",...","{\n ""object_#1"": ""the first basket is longer ...","{\n ""object_#1"": ""longer than the rest, kin..."
...,...,...,...,...,...,...,...
75,dogs,4,Pair_34,D: The first dog is sittin' on the grass\nThe...,"{\n ""object_#1"": ""sittin' on the grass"",\n ...","{\n ""object_#1"": ""sittin' on the grass, The...","{\n ""object_#1"": ""sittin' on the grass, old..."
76,dogs,4,Pair_35,"D: One\nUh, tail sticking up in the air\n\nM:...","{\n ""object_#1"": ""tail sticking up in the a...","{\n ""object_#1"": ""tail sticking up in the a...","{\n ""object_#1"": ""tail sticking up in the a..."
77,dogs,4,Pair_37,"D: Uh, this is the one with the...the hairy c...","{\n ""object_#1"": ""the hairy chin and the ta...","{\n ""object_#1"": ""hairy chin, tail stickin'...","{\n ""object_#1"": ""hairy chin, tail stickin'..."
78,dogs,4,Pair_43,"D: Number one--um, it's Toto\n\nM: Um, ok\n\nD...","{\n ""object_#1"": ""it's Toto"",\n ""object_...","{\n ""object_#1"": ""it's Toto"",\n ""object_...","{\n ""object_#1"": ""it's Toto"",\n ""object_..."


In [4]:
import json


def parse_llm_response(response):
    response = response.replace("json", "").strip()
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        parsed = []
    return parsed


df["parsed_llm_response"] = df["llm_extracted_object_descriptions_GPT_5"].apply(parse_llm_response)    

In [5]:
df["parsed_llm_response"].apply(len).describe()

count    80.0
mean     10.0
std       0.0
min      10.0
25%      10.0
50%      10.0
75%      10.0
max      10.0
Name: parsed_llm_response, dtype: float64

In [6]:
dogs_df = pd.read_excel("baskets_dogs_data/dogs.xlsx")
baskets_df = pd.read_excel("baskets_dogs_data/baskets.xlsx")

In [7]:
dogs_df.columns, baskets_df.columns

(Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
        'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
        'Pair_45'],
       dtype='object'),
 Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
        'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
        'Pair_45'],
       dtype='object'))

In [8]:
dogs_df[dogs_df["Round"] == 1]["TargetObjectIndex"].tolist()

[2, 8, 4, 10, 6, 3, 7, 1, 9, 5]

In [9]:
FirstRoundCardIndex_map = {}

FirstRoundCardIndex_map["dogs"] = {
    target_obj_ix: f"Card{i+1}" for i, target_obj_ix in enumerate(dogs_df[dogs_df["Round"] == 1]["TargetObjectIndex"].tolist())
}
FirstRoundCardIndex_map["baskets"] = {
    target_obj_ix: f"Card{i+1}" for i, target_obj_ix in enumerate(baskets_df[baskets_df["Round"] == 1]["TargetObjectIndex"].tolist())
}
FirstRoundCardIndex_map

{'dogs': {2: 'Card1',
  8: 'Card2',
  4: 'Card3',
  10: 'Card4',
  6: 'Card5',
  3: 'Card6',
  7: 'Card7',
  1: 'Card8',
  9: 'Card9',
  5: 'Card10'},
 'baskets': {9: 'Card1',
  3: 'Card2',
  7: 'Card3',
  5: 'Card4',
  10: 'Card5',
  1: 'Card6',
  4: 'Card7',
  6: 'Card8',
  2: 'Card9',
  8: 'Card10'}}

In [10]:
extracted_descriptions = []
cols = ["pair_id", "round_ix", "transcript", "object_ix", "filename", "object_description"]

for _, row in df.iterrows():
    pair_id = row["pair_id"]
    round_ix = row["round_ix"]
    transcript = row["transcript"]
    object_category = row["object_category"]
    description_dict = row["parsed_llm_response"]

    if object_category == "dogs":
        target_object_ixes = dogs_df[dogs_df.Round == round_ix].TargetObjectIndex.to_list()
    elif object_category == "baskets":
        target_object_ixes = baskets_df[baskets_df.Round == round_ix].TargetObjectIndex.to_list()
    else:
        print(f"Unknown object category: {object_category} in pair {pair_id}, round {round_ix}")
        continue

    for i, key in enumerate(description_dict):
        object_ix = i+1
        
        if f"object_#{object_ix}" != key:
            print(
                f"Warning: Expected object_#{object_ix} but got {key} in pair {pair_id}, round {round_ix}"
            )
            continue
        
        object_description = description_dict.get(key, None)

        extracted_descriptions.append(
            {
                "object_category": object_category,
                "pair_id": pair_id,
                "round_ix": round_ix,
                "transcript": transcript,
                "FirstRoundCardIndex": FirstRoundCardIndex_map[object_category][target_object_ixes[i]],
                "target_object_ix": target_object_ixes[i],
                "object_description": object_description,
            }
        )

In [11]:
extracted_descriptions_df = pd.DataFrame(extracted_descriptions)
extracted_descriptions_df

,object_category,pair_id,round_ix,transcript,FirstRoundCardIndex,target_object_ix,object_description
0,baskets,Pair_20,1,"D: Alright, the first one is a, uh--is a very ...",Card1,9,"a very long basket, rectangular in shape, ligh..."
1,baskets,Pair_20,1,"D: Alright, the first one is a, uh--is a very ...",Card2,3,"got a huge handle, a very small basket, red, a..."
2,baskets,Pair_20,1,"D: Alright, the first one is a, uh--is a very ...",Card3,7,"a large basket, another handle, the same color..."
3,baskets,Pair_20,1,"D: Alright, the first one is a, uh--is a very ...",Card4,5,"a very tight weave, without a handle, a very l..."
4,baskets,Pair_20,1,"D: Alright, the first one is a, uh--is a very ...",Card5,10,"a small basket, with a wide weave, a red weave..."
...,...,...,...,...,...,...,...
795,dogs,Pair_45,4,"D: Um, Joanne, number one is the dog with the...",Card5,6,the poodle
796,dogs,Pair_45,4,"D: Um, Joanne, number one is the dog with the...",Card4,10,the hotdog
797,dogs,Pair_45,4,"D: Um, Joanne, number one is the dog with the...",Card3,4,it's sitting down
798,dogs,Pair_45,4,"D: Um, Joanne, number one is the dog with the...",Card2,8,the dog with one leg extended out


In [12]:
extracted_descriptions_df_sorted = []


for object_category in extracted_descriptions_df["object_category"].unique():
    for pair_id in extracted_descriptions_df["pair_id"].unique():
        subset = extracted_descriptions_df[
            (extracted_descriptions_df["object_category"] == object_category)
            & (extracted_descriptions_df["pair_id"] == pair_id)
        ]
        if subset.empty:
            continue
        
        for FirstRoundCardIndex in subset["FirstRoundCardIndex"].unique():
            subset_card = subset[subset["FirstRoundCardIndex"] == FirstRoundCardIndex]
            if subset_card.empty:
                continue
            
            row = {
                "object_category": object_category,
                "pair_ix": int(pair_id.replace("Pair_", "")),
                "FirstRoundCardIndex": FirstRoundCardIndex,
            }
            for round_ix in range(1, 5):
                desc = subset_card[subset_card["round_ix"] == round_ix]["object_description"]
                row[f"Round{round_ix}"] = desc.values[0] if not desc.empty else None
            
            extracted_descriptions_df_sorted.append(row)

extracted_descriptions_sorted_df = pd.DataFrame(extracted_descriptions_df_sorted)
extracted_descriptions_sorted_df

,object_category,pair_ix,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,baskets,20,Card1,"a very long basket, rectangular in shape, ligh...","the rectangular large, Very long","really long, rectangular one, handle in the mi...",Rectangle
1,baskets,20,Card2,"got a huge handle, a very small basket, red, a...","the one with the very large handle, Very small...","small, large handle",small basket
2,baskets,20,Card3,"a large basket, another handle, the same color...","very round, with the very open weave, it looks...","rounded one, very open weave, the black in the...","very open weave, black bottom"
3,baskets,20,Card4,"a very tight weave, without a handle, a very l...",the very large,"very large bowl, tight weave","very large bowl, tight weave"
4,baskets,20,Card5,"a small basket, with a wide weave, a red weave...",the red ribbon striped one,"basket with the red ribbon, running through it",red ribbon
...,...,...,...,...,...,...,...
195,dogs,45,Card6,"a German shepherd, it's black, in the nose par...","a German shepherd, it's standing on the grass,...",the German shepherd,the German shepherd
196,dogs,45,Card7,"a golden retriever, laying down on the grass, ...",the golden retriever that's laying on the ground,"the one that's laying down, The golden retriever","the golden retriever that's laying down, Long ..."
197,dogs,45,Card8,"goldish, the mouth and nose part is black, has...","dog with the black nose and mouth, the rest of...","black on the mouth, two legs extending out, it...","the dog with the two legs extending out, goldi..."
198,dogs,45,Card9,"a wolf, I think, a white dog, blackish on the ...","the white dog, the wolf","the white dog, the wolf",the white dog


In [13]:
annotated_df = pd.read_csv("baskets_dogs_data/entrainment_coding_annotated_data.csv")
test_df = annotated_df[annotated_df.split == "test"]

In [16]:
import re


STOP_WORDS = ['a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 
              'and', 'any', 'are', "aren't", 'as', 'at', 'be', 'because', 'been', 
              'before', 'being', 'below', 'between', 'both', 'but', 'by', "can't", 
              'cannot', 'could', "couldn't", 'did', "didn't", 'do', 'does', "doesn't", 
              'doing', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 
              'had', "hadn't", 'has', "hasn't", 'have', "haven't", 'having', 'he', "he'd", 
              "he'll", "he's", 'her', 'here', "here's", 'hers', 'herself', 'him', 'himself', 
              'his', 'how', "how's", 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 
              "isn't", 'it', "it's", 'its', 'itself', "let's", 'me', 'more', 'most', "mustn't", 'my', 
              'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 
              'our', 'ours', 'ourselves', 'out', 'over', 'own', 'same', "shan't", 'she', "she'd", "she'll", 
              "she's", 'should', "shouldn't", 'so', 'some', 'such', 'than', 'that', "that's", 'the', 'their', 
              'theirs', 'them', 'themselves', 'then', 'there', "there's", 'these', 'they', "they'd", "they'll", 
              "they're", "they've", 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 
              'was', "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', "weren't", 'what', "what's", 'when', 
              "when's", 'where', "where's", 'which', 'while', 'who', "who's", 'whom', 'why', "why's", 'with', "won't", 
              'would', "wouldn't", 'you', "you'd", "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves']

def preprocess_text(text: str) -> str:
    # remove punctuation and lowercase
    text = re.sub(r"[^\w\s]", "", text.lower())

    if not isinstance(text, str):
        return ""
    
    tokens = re.findall(r"\b\w+\b", text)
    filtered = [t for t in tokens if t not in STOP_WORDS]
    return " ".join(filtered)

In [17]:
for i in range(1, 5):
    extracted_descriptions_sorted_df[f"Round{i}_preprocessed"] = extracted_descriptions_sorted_df[f"Round{i}"].apply(preprocess_text)
    test_df[f"Round{i}_preprocessed"] = test_df[f"Round{i}"].apply(preprocess_text)

/tmp/ipykernel_1833328/3241040847.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"Round{i}_preprocessed"] = test_df[f"Round{i}"].apply(preprocess_text)
/tmp/ipykernel_1833328/3241040847.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"Round{i}_preprocessed"] = test_df[f"Round{i}"].apply(preprocess_text)
/tmp/ipykernel_1833328/3241040847.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

### Evaluation

In [18]:
from rouge import Rouge
from sentence_transformers import SentenceTransformer, util

def rouge_f1(system_summary: str, reference_summary: str):
    rouge = Rouge()
    scores = rouge.get_scores(system_summary, reference_summary)[0]

    r1_f1 = scores["rouge-1"]["f"]
    r2_f1 = scores["rouge-2"]["f"]
    rL_f1 = scores["rouge-l"]["f"]

    return r1_f1, r2_f1, rL_f1


# Load once (global) so you don't re-load the model on every call
_sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def sbert_score(text1: str, text2: str) -> float:
    """
    Compute SBERT cosine similarity between two strings.
    Returns a single float (typically in [0, 1] for normal sentences).
    """
    emb1 = _sbert_model.encode(text1, convert_to_tensor=True)
    emb2 = _sbert_model.encode(text2, convert_to_tensor=True)

    # util.cos_sim returns a 1x1 tensor here
    score = util.cos_sim(emb1, emb2).item()
    return float(score)

In [19]:
test_df.head(1)

,object_category,pair_ix,split,FirstRoundCardIndex,Round1,Round2,Round3,Round4,Round1_preprocessed,Round2_preprocessed,Round3_preprocessed,Round4_preprocessed
0,baskets,45,test,Card1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped with handle in middle, no lid",rectangle shaped with handle in middle,rectangular shaped with handle in middle,rectangle shaped handle middle lid empty,rectangle shaped handle middle lid,rectangle shaped handle middle,rectangular shaped handle middle


In [20]:
extracted_descriptions_sorted_df.head(1)

,object_category,pair_ix,FirstRoundCardIndex,Round1,Round2,Round3,Round4,Round1_preprocessed,Round2_preprocessed,Round3_preprocessed,Round4_preprocessed
0,baskets,20,Card1,"a very long basket, rectangular in shape, ligh...","the rectangular large, Very long","really long, rectangular one, handle in the mi...",Rectangle,long basket rectangular shape light tan weaved...,rectangular large long,really long rectangular one handle middle,rectangle


In [21]:
evaluation_results = []

for object_category in test_df.object_category.unique():
    test_df_sub = test_df[test_df.object_category == object_category]
    annotated_llm_df_sub = extracted_descriptions_sorted_df[extracted_descriptions_sorted_df.object_category == object_category]

    for pair_ix in test_df_sub.pair_ix.unique():
        test_df_subsub = test_df_sub[test_df_sub.pair_ix == pair_ix]
        annotated_llm_df_subsub = annotated_llm_df_sub[annotated_llm_df_sub.pair_ix == pair_ix]

        for ix, row in test_df_subsub.iterrows():
            FirstRoundCardIndex = row.FirstRoundCardIndex
            annotated_row = annotated_llm_df_subsub[
                annotated_llm_df_subsub.FirstRoundCardIndex == FirstRoundCardIndex
            ].iloc[0]

            for round_num in range(1, 5):
                reference_summary = row[f"Round{round_num}_preprocessed"]
                system_summary = annotated_row[f"Round{round_num}_preprocessed"]

                if system_summary == "":
                    system_summary = "dummy_summary"

                r1_f1, r2_f1, rL_f1 = rouge_f1(system_summary, reference_summary)

                evaluation_results.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "FirstRoundCardIndex": FirstRoundCardIndex,
                    "round_num": round_num,
                    "rouge-1_f1": r1_f1,
                    "rouge-2_f1": r2_f1,
                    "rouge-L_f1": rL_f1,
                    "sbert_score": sbert_score(system_summary, reference_summary),
                })

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

,object_category,pair_ix,FirstRoundCardIndex,round_num,rouge-1_f1,rouge-2_f1,rouge-L_f1,sbert_score
0,baskets,45,Card1,1,0.727273,0.444444,0.727273,0.605221
1,baskets,45,Card1,2,0.800000,0.500000,0.800000,0.804606
2,baskets,45,Card1,3,0.888889,0.571429,0.888889,0.969610
3,baskets,45,Card1,4,1.000000,1.000000,1.000000,1.000000
4,baskets,45,Card2,1,0.583333,0.560000,0.583333,0.826125
...,...,...,...,...,...,...,...,...
795,dogs,24,Card9,4,0.857143,0.800000,0.857143,0.925368
796,dogs,24,Card10,1,0.909091,0.761905,0.909091,0.822594
797,dogs,24,Card10,2,0.909091,0.777778,0.909091,0.982311
798,dogs,24,Card10,3,1.000000,1.000000,1.000000,1.000000


In [24]:
evaluation_df.columns

Index(['object_category', 'pair_ix', 'FirstRoundCardIndex', 'round_num',
       'rouge-1_f1', 'rouge-2_f1', 'rouge-L_f1', 'sbert_score'],
      dtype='object')

In [29]:
evaluation_df[["rouge-1_f1", "rouge-2_f1", "rouge-L_f1", "sbert_score"]].describe().round(2)

,rouge-1_f1,rouge-2_f1,rouge-L_f1,sbert_score
count,800.00,800.00,800.00,800.00
mean,0.86,0.65,0.86,0.90
std,0.15,0.32,0.15,0.12
min,0.00,0.00,0.00,0.05
25%,0.80,0.44,0.80,0.83
50%,0.89,0.69,0.89,0.93
75%,1.00,1.00,1.00,1.00
max,1.00,1.00,1.00,1.00


In [32]:
print(evaluation_df[["rouge-L_f1", "sbert_score"]].agg(["mean", "std"]).to_latex(float_format="%.2f"))

\begin{tabular}{lrr}
\toprule
 & rouge-L_f1 & sbert_score \\
\midrule
mean & 0.86 & 0.90 \\
std & 0.15 & 0.12 \\
\bottomrule
\end{tabular}



In [25]:
evaluation_df.groupby("round_num")[["rouge-1_f1", "rouge-2_f1", "rouge-L_f1", "sbert_score"]].describe().T

round_num                   1           2           3           4
rouge-1_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.832877    0.857731    0.848614    0.905472
            std      0.133686    0.155807    0.175807    0.132343
            min      0.333333    0.000000    0.000000    0.285714
            25%      0.750000    0.795652    0.800000    0.857143
            50%      0.857143    0.892720    0.888889    1.000000
            75%      0.933333    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
rouge-2_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.615867    0.667978    0.634758    0.687714
            std      0.256304    0.302832    0.343613    0.360527
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.444444    0.500000    0.400000    0.500000
            50%      0.625000    0.727273    0.666667    0.800000
            75%      0.800000    0.925641    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
rouge-L_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.827757    0.856303    0.848229    0.904222
            std      0.137659    0.157117    0.176066    0.134974
            min      0.333333    0.000000    0.000000    0.285714
            25%      0.747685    0.781401    0.800000    0.857143
            50%      0.841053    0.892720    0.888889    1.000000
            75%      0.933333    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
sbert_score count  200.000000  200.000000  200.000000  200.000000
            mean     0.871141    0.889371    0.896671    0.924461
            std      0.105614    0.133095    0.131092    0.112698
            min      0.535807    0.051621    0.101779    0.450619
            25%      0.802129    0.829761    0.841779    0.877257
            50%      0.894802    0.931484    0.939177    0.986384
            75%      0.955792    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000

In [26]:
evaluation_df.groupby("round_num")[["rouge-1_f1", "rouge-2_f1", "rouge-L_f1", "sbert_score"]].describe().T

round_num                   1           2           3           4
rouge-1_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.832877    0.857731    0.848614    0.905472
            std      0.133686    0.155807    0.175807    0.132343
            min      0.333333    0.000000    0.000000    0.285714
            25%      0.750000    0.795652    0.800000    0.857143
            50%      0.857143    0.892720    0.888889    1.000000
            75%      0.933333    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
rouge-2_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.615867    0.667978    0.634758    0.687714
            std      0.256304    0.302832    0.343613    0.360527
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.444444    0.500000    0.400000    0.500000
            50%      0.625000    0.727273    0.666667    0.800000
            75%      0.800000    0.925641    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
rouge-L_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.827757    0.856303    0.848229    0.904222
            std      0.137659    0.157117    0.176066    0.134974
            min      0.333333    0.000000    0.000000    0.285714
            25%      0.747685    0.781401    0.800000    0.857143
            50%      0.841053    0.892720    0.888889    1.000000
            75%      0.933333    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000
sbert_score count  200.000000  200.000000  200.000000  200.000000
            mean     0.871141    0.889371    0.896671    0.924461
            std      0.105614    0.133095    0.131092    0.112698
            min      0.535807    0.051621    0.101779    0.450619
            25%      0.802129    0.829761    0.841779    0.877257
            50%      0.894802    0.931484    0.939177    0.986384
            75%      0.955792    1.000000    1.000000    1.000000
            max      1.000000    1.000000    1.000000    1.000000

In [21]:
combined_df = []
cols = ["object_category", "pair_ix", "FirstRoundCardIndex", "round_ix", "human_extracted", "llm_extracted"]

for object_category in test_df.object_category.unique():
    test_df_sub = test_df[test_df.object_category == object_category]
    annotated_llm_df_sub = extracted_descriptions_sorted_df[extracted_descriptions_sorted_df.object_category == object_category]

    for pair_ix in test_df_sub.pair_ix.unique():
        test_df_subsub = test_df_sub[test_df_sub.pair_ix == pair_ix]
        annotated_llm_df_subsub = annotated_llm_df_sub[annotated_llm_df_sub.pair_ix == pair_ix]

        for ix, row in test_df_subsub.iterrows():
            FirstRoundCardIndex = row.FirstRoundCardIndex
            annotated_row = annotated_llm_df_subsub[
                annotated_llm_df_subsub.FirstRoundCardIndex == FirstRoundCardIndex
            ].iloc[0]

            for round_ix in range(1, 5):
                human_extracted = row[f"Round{round_ix}"]
                llm_extracted = annotated_row[f"Round{round_ix}"]

                combined_df.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "FirstRoundCardIndex": FirstRoundCardIndex,
                    "round_ix": round_ix,
                    "human_extracted": human_extracted,
                    "llm_extracted": llm_extracted,
                })

combined_df = pd.DataFrame(combined_df)
combined_df

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
0,baskets,45,Card1,1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped basket, the handle in the middle"
1,baskets,45,Card1,2,"rectangle shaped with handle in middle, no lid",the rectangle shaped one with the handle in th...
2,baskets,45,Card1,3,rectangle shaped with handle in middle,the rectangle shaped one with the handle in th...
3,baskets,45,Card1,4,rectangular shaped with handle in middle,the rectangular shaped with the handle in the ...
4,baskets,45,Card2,1,"Easter basket, pink things, pink wood, twisted...","a Easter basket, pink…things, Pink wood, a twi..."
...,...,...,...,...,...,...
795,dogs,24,Card9,4,black and white husky,
796,dogs,24,Card10,1,hair drapes all the way from his body to groun...,his hair drapes all the way from his body to t...
797,dogs,24,Card10,2,"terrier, all black, longest haired dog, short ...","a terrier, All black, this is the longest hair..."
798,dogs,24,Card10,3,short little stumpy dog with moppy hair,"The black one, the short little stumpy dog wit..."


In [22]:
from src.data.utils import pretty_print

pretty_print(combined_df.sample(5))

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
752,dogs,34,Card9,1,"the only white dog, with a grayish--.","It's the only white dog, White with a grayish t--"
374,baskets,24,Card4,3,"wide, deep, no handle, no lid, tight weaving","wide, kinda deep, no handle, no lid, Tight weaving"
339,baskets,34,Card5,4,red tape around,basket with the red tape around it
575,dogs,23,Card4,4,weiner,the weiner dog?
101,baskets,22,Card6,2,"green triangles, green through back, designs","the green triangles one, and those...designs on the--"


In [23]:
pretty_print(combined_df.sample(5))

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
136,baskets,20,Card5,1,"small, wide weave, red weave through whole thing, red ribbon","small basket with a wide weave, it's got a red weave through the whole thing"
723,dogs,34,Card1,4,with black middle and black little tail,with the black middle and the black little tail
438,dogs,45,Card10,3,"fur hanging out of mouth, and sitting on grass with tail sticking up",the one with the fur hanging out of its mouth and sitting on the grass with the tail sticking up
213,baskets,37,Card4,2,"big round, with no lid, no handles","the big round one with no lid, no handles"
744,dogs,34,Card7,1,"golden retriever, gold, has long ears, facing upwards to the left, laying on grass or something","it's gold, It has long ears, facing to the left, he's layin' on grass or something"


In [24]:
pretty_print(combined_df.sample(10))

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
72,baskets,35,Card9,1,"flat top, round handle, round basket, very simple, handle is more rectangle shape, goes all the way up to top, goes over whole thing, just plain round","flat top, round handle, round basket, very simple, handle is more like rectangle shape, Goes all the way up to the top, Goes over the whole thing, round, Just plain round, simple, Flat top"
133,baskets,20,Card4,2,very large,"the round basket with the round top, it's got a square handle"
63,baskets,35,Card6,4,green line,has a green line
281,baskets,43,Card1,2,"narrow, long, handle, no cover","it's narrow, it's long, it has a handle, No cover"
159,baskets,20,Card10,4,"tapered, rough lid","the tapered, with the rough lid"
674,dogs,21,Card9,3,Siberian husky,the Siberian husky
192,baskets,23,Card9,1,handle looks like half a square,the handle looks sort of like half of a square
534,dogs,20,Card4,3,weiner dog,the weiner dog
731,dogs,34,Card3,4,sitting in grass with head turned to your left,the dog sittin' in the grass with his head turned to *your left*
368,baskets,24,Card3,1,"handle, spaces in between weavings, wide, not tight, looks like black thing inside of it","the third basket has a handle, spaces in between the weavings, they're wide, They're not tight, looks like there's a black thing inside of it"
